# 01 · Download dataset

Downloads one Roboflow dataset version twice, into folders named `<project>-v<version>-<format>`:
- **COCO segmentation** → `datasets/leaf-dis-seg-v1-coco/`: ground truth for `03_evaluate.ipynb` and training data for RF-DETR.
- **YOLO segmentation** (polygon labels + `data.yaml`) → `datasets/leaf-dis-seg-v1-yolo/`: training data for YOLO.

Reads `ROBOFLOW_API_KEY`, `ROBOFLOW_WORKSPACE`, `ROBOFLOW_PROJECT`, `ROBOFLOW_VERSION` from `.env`. Run from the `training/` folder.
For a new data version, set `ROBOFLOW_VERSION=2` in `.env` and rerun: v2 lands in its own `-v2-` folders and v1 stays untouched.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import yaml
from dotenv import load_dotenv

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")

DATA_NAME = f"{os.environ['ROBOFLOW_PROJECT']}-v{os.environ['ROBOFLOW_VERSION']}"  # e.g. leaf-dis-seg-v1
DATASET_DIR = ROOT / "datasets" / f"{DATA_NAME}-coco"
YOLO_DIR = ROOT / "datasets" / f"{DATA_NAME}-yolo"
print(DATASET_DIR, YOLO_DIR, sep="\n")

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
version = (
    rf.workspace(os.environ["ROBOFLOW_WORKSPACE"])
    .project(os.environ["ROBOFLOW_PROJECT"])
    .version(int(os.environ["ROBOFLOW_VERSION"]))
)
version.download("coco-segmentation", location=str(DATASET_DIR))
version.download("yolov8", location=str(YOLO_DIR))  # same polygon format is used by YOLO11 / YOLO26 seg

## Fix `data.yaml` paths

Roboflow writes split paths like `../train/images`, which Ultralytics resolves from the wrong folder. Point them at this dataset folder instead.

In [ ]:
data_yaml = YOLO_DIR / "data.yaml"
data = yaml.safe_load(data_yaml.read_text())
data.update(path=str(YOLO_DIR), train="train/images", val="valid/images", test="test/images")
data_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
print(data_yaml.read_text())

In [ ]:
# Drop box-only label lines (class + 4 numbers). In a segmentation export these are 1–2 px specks (accidental clicks),
# and a mix of boxes and polygons makes Ultralytics drop every mask and train boxes only.
removed = 0
for lbl in YOLO_DIR.glob("*/labels/*.txt"):
    lines = lbl.read_text().splitlines()
    keep = [ln for ln in lines if len(ln.split()) != 5]
    if len(keep) != len(lines):
        removed += len(lines) - len(keep)
        lbl.write_text("\n".join(keep) + ("\n" if keep else ""))
print("removed box-only label lines:", removed)

## Check splits and class balance

Expect `leaf`, `leaf_spot`, `powdery_mildew` (Roboflow may also add a parent category with no annotations to the COCO export).

In [ ]:
for split in ("train", "valid", "test"):
    coco = json.loads((DATASET_DIR / split / "_annotations.coco.json").read_text())
    names = {c["id"]: c["name"] for c in coco["categories"]}
    counts = Counter(names[a["category_id"]] for a in coco["annotations"])
    n_yolo = len(list((YOLO_DIR / split / "images").iterdir()))
    print(f"{split:>5}: {len(coco['images']):4d} images (YOLO: {n_yolo})  {dict(counts)}")

## Look at a few training images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pycocotools.coco import COCO

COLORS = {"leaf": (0, 200, 0), "leaf_spot": (230, 120, 0), "powdery_mildew": (180, 0, 220)}

train = COCO(str(DATASET_DIR / "train" / "_annotations.coco.json"))
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img in zip(axes, train.loadImgs(train.getImgIds()[:4])):
    image = np.array(Image.open(DATASET_DIR / "train" / img["file_name"]).convert("RGB"), dtype=float)
    for ann in train.loadAnns(train.getAnnIds(imgIds=img["id"])):
        color = COLORS.get(train.cats[ann["category_id"]]["name"], (255, 255, 0))
        m = train.annToMask(ann).astype(bool)
        image[m] = 0.5 * image[m] + 0.5 * np.array(color)
    ax.imshow(image.astype(np.uint8))
    ax.set_title(img["file_name"][:30])
    ax.axis("off")